## Agrupamentos por bairro

In [1]:
import pandas as pd
import folium
from setup_notebook import setup_path
setup_path()
from src.utils.functions import *
from folium.plugins import FastMarkerCluster
import webbrowser
import os
import time
from unidecode import unidecode


In [2]:
#leitura do arquivo
arquivo_csv = '/home/akel/PycharmProjects/city_noise/data/processed/Bares_etc_Belem_filt.csv'


# 2. Ler o arquivo CSV
df = pd.read_csv(arquivo_csv, sep=',', encoding='utf-8', low_memory=False)

In [3]:
# 2. Fazer a contagem de estabelecimentos por Bairro
contagem_por_bairro = df['BAIRRO'].value_counts()
contagem_por_bairro.head(30)

BAIRRO
GUAMA                    202
MARCO                    187
JURUNAS                  168
TAPANA                   160
PEDREIRA                 156
MARAMBAIA                155
COQUEIRO                 132
CONDOR                   131
CRUZEIRO                 129
SACRAMENTA               127
SAO JOAO DO OUTEIRO      121
CAMPINA DE ICOARACI      115
TELEGRAFO                111
BENGUI                    94
UMARIZAL                  94
MANGUEIRAO                88
PARQUE VERDE              85
MONTESE (TERRA FIRME)     84
CAMPINA                   84
SAO BRAS                  80
AGUA BOA                  80
CASTANHEIRA               80
CREMACAO                  68
CIDADE VELHA              63
CURIO-UTINGA              61
NAZARE                    60
MARACANGALHA              58
PARQUE GUAJARA            53
PONTA GROSSA              51
BATISTA CAMPOS            51
Name: count, dtype: int64

In [4]:
contagem_por_distrito = df['DISTRITO'].value_counts()
contagem_por_distrito

DISTRITO
DAGUA    691
DABEL    641
DABEN    607
DASAC    527
DAICO    523
DAENT    487
DAMOS    317
DAOUT    260
Name: count, dtype: int64

In [5]:

# 1. Garante a formatação do texto antes de categorizar
df['DSC_ESTABELECIMENTO'] = df['DSC_ESTABELECIMENTO'].astype(str).str.upper()

padrao_bar = r'\bBAR\b|\bBOTECO\b'
padrao_deposito = r'DEPOSITO.*BEBIDA|DEPOSITO.*CERVEJA'
padrao_sinuca = r'\bSINUCA\b|\bBILHAR\b'
padrao_restaurante = r'CHURRASC|RESTAURANTE|PEIXARIA' 
padrao_shows = r'CASA DE SHOW|CASA DE EVENTOS|RECEPCO' # Ajustado sem acento
padrao_cafe = r'CAFE|TAPIOCA'

# 3. Condições (Subi RESTAURANTE para antes de BAR caso queira priorizá-lo em "Bar e Restaurante")
# Se preferir que "Bar e Restaurante" vire BAR, basta voltar o padrao_bar para a primeira posição.
condicoes = [
    df['DSC_ESTABELECIMENTO'].str.contains(padrao_restaurante, na=False, regex=True),
    df['DSC_ESTABELECIMENTO'].str.contains(padrao_bar, na=False, regex=True),
    df['DSC_ESTABELECIMENTO'].str.contains(padrao_deposito, na=False, regex=True),
    df['DSC_ESTABELECIMENTO'].str.contains(padrao_sinuca, na=False, regex=True),
    df['DSC_ESTABELECIMENTO'].str.contains(padrao_shows, na=False, regex=True),
    df['DSC_ESTABELECIMENTO'].str.contains(padrao_cafe, na=False, regex=True)
]

# 4. Categorias equivalentes (Mantendo a mesma ordem das condições acima!)
categorias = [
    'RESTAURANTE',
    'BAR',
    'DEPOSITO DE BEBIDAS',
    'CASA DE JOGOS',
    'CASA DE SHOW',
    'CAFE'
]

# 5. Aplicação do filtro
df['TIPO_ESTABELECIMENTO'] = np.select(condicoes, categorias, default='OUTROS')
contagem_por_TIPO = df['TIPO_ESTABELECIMENTO'].value_counts()
contagem_por_TIPO


TIPO_ESTABELECIMENTO
BAR                    1883
RESTAURANTE            1582
DEPOSITO DE BEBIDAS     347
CAFE                    185
CASA DE SHOW             47
CASA DE JOGOS             9
Name: count, dtype: int64

In [36]:
# #6) VISUALIZAÇÃO NO MAPA
# centro_lat = df['LATITUDE'].dropna().mean()
# centro_lon = df['LONGITUDE'].dropna().mean()
# centro = [centro_lat, centro_lon]

# mapa = folium.Map(location=centro, zoom_start=12)

# for _, row in df.dropna(subset=['LATITUDE', 'LONGITUDE']).iterrows():
#     folium.CircleMarker(
#         location=[row["LATITUDE"], row["LONGITUDE"]],
#         radius=4,
#         color="red",
#         fill=True,
#         fill_color="red",
#         # Opcional: adiciona o nome do estabelecimento como popup ao clicar
#         popup=row["DSC_ESTABELECIMENTO"] 
#     ).add_to(mapa)

# # Mostrar mapa
# mapa

In [6]:
df_contagem = (
    df.groupby(['BAIRRO', 'TIPO_ESTABELECIMENTO'])
    .size()
    .reset_index(name='QUANTIDADE')
)

top5_por_tipo = (
    df_contagem.sort_values(by=['TIPO_ESTABELECIMENTO', 'QUANTIDADE'], ascending=[True, False])
    .groupby('TIPO_ESTABELECIMENTO')
    .head(5)
)

# 3. Exibe o resultado formatado como uma lista limpa
print("=== TOP 5 BAIRROS POR TIPO DE ESTABELECIMENTO ===")

for tipo in top5_por_tipo['TIPO_ESTABELECIMENTO'].unique():
    print(f"\n📌 {tipo}")
    print("-" * 40)
    
    # Isola os dados do tipo atual
    sub_df = top5_por_tipo[top5_por_tipo['TIPO_ESTABELECIMENTO'] == tipo]
    
    # Loop para listar a posição (1º ao 5º)
    for i, linha in enumerate(sub_df.itertuples(), 1):
        print(f"{i}º. {linha.BAIRRO} ({linha.QUANTIDADE} locais)")

=== TOP 5 BAIRROS POR TIPO DE ESTABELECIMENTO ===

📌 BAR
----------------------------------------
1º. GUAMA (87 locais)
2º. JURUNAS (85 locais)
3º. SAO JOAO DO OUTEIRO (84 locais)
4º. MARCO (73 locais)
5º. TAPANA (72 locais)

📌 CAFE
----------------------------------------
1º. JURUNAS (9 locais)
2º. MARCO (9 locais)
3º. NAZARE (9 locais)
4º. UMARIZAL (9 locais)
5º. BENGUI (8 locais)

📌 CASA DE JOGOS
----------------------------------------
1º. SAO CLEMENTE (2 locais)
2º. CASTANHEIRA (1 locais)
3º. CONDOR (1 locais)
4º. COQUEIRO (1 locais)
5º. CRUZEIRO (1 locais)

📌 CASA DE SHOW
----------------------------------------
1º. MANGUEIRAO (4 locais)
2º. TELEGRAFO (4 locais)
3º. UMARIZAL (4 locais)
4º. PEDREIRA (3 locais)
5º. CAMPINA (2 locais)

📌 DEPOSITO DE BEBIDAS
----------------------------------------
1º. JURUNAS (28 locais)
2º. GUAMA (27 locais)
3º. SACRAMENTA (19 locais)
4º. MONTESE (TERRA FIRME) (17 locais)
5º. CAMPINA DE ICOARACI (16 locais)

📌 RESTAURANTE
--------------------------

In [20]:
dfc=df[df['TIPO_ESTABELECIMENTO']=='DEPOSITO DE BEBIDAS']
df[ ( df['TIPO_ESTABELECIMENTO'] == 'CAFE') & (df['BAIRRO'] == 'NAZARE')]

,DISTRITO,BAIRRO,END_COMPLETO,NUM_ENDERECO,LATITUDE,LONGITUDE,DSC_ESTABELECIMENTO,COD_INDICADOR_ESTAB_ENDERECO,TIPO_ESTABELECIMENTO
816,DABEL,NAZARE,AVENIDA GENERALISSIMO DEODORO,1071,-1.450217,-48.482772,CAFETERIA EMPORIO AVENIDA,1.0,CAFE
817,DABEL,NAZARE,AVENIDA GOVERNADOR JOSE MALCHER,1350,-1.449819,-48.481947,CAFETERIA SODIE DOCES,1.0,CAFE
1244,DABEL,NAZARE,AVENIDA GENERALISSIMO DEODORO,1533,-1.454473,-48.482705,CAFE BISTRO POINT GOURMET,1.0,CAFE
1882,DABEL,NAZARE,AVENIDA SERZEDELO CORREA,15,-1.454253,-48.492515,CAFE,1.0,CAFE
2315,DABEL,NAZARE,TRAVESSA QUATORZE DE MARCO,1933,-1.453169,-48.480546,CHEIA DE GRACA CAFE,1.0,CAFE
3104,DABEL,NAZARE,TRAVESSA DOUTOR MORAES,49,-1.454157,-48.490823,CAFE COM AMOR,1.0,CAFE
3698,DABEL,NAZARE,ALAMEDA PEDRO CARNEIRO,10,-1.455698,-48.479395,MUNDICA TAPIOCARIA E IGUARIAS,1.0,CAFE
3736,DABEL,NAZARE,TRAVESSA RUI BARBOSA,1437,-1.453971,-48.487104,ENTRETERIMENTO CAFE COM ARTE,1.0,CAFE
4018,DABEL,NAZARE,AVENIDA COMANDANTE BRAS DE AGUIAR,948,-1.454265,-48.483067,BITITA BRISTO E CAFE,1.0,CAFE


In [21]:
#6) VISUALIZAÇÃO NO MAPA


centro_lat = df['LATITUDE'].dropna().mean()
centro_lon = df['LONGITUDE'].dropna().mean()
centro = [centro_lat, centro_lon]

mapa = folium.Map(location=centro, zoom_start=12)

for _, row in df.dropna(subset=['LATITUDE', 'LONGITUDE']).iterrows():
    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=4,
        color="red",
        fill=True,
        fill_color="red",
        # Opcional: adiciona o nome do estabelecimento como popup ao clicar
        popup=row["DSC_ESTABELECIMENTO"] 
    ).add_to(mapa)

# Mostrar mapa
mapa

In [24]:

# 1. Dicionário mapeando cada categoria a uma cor específica do Folium
cores_categorias = {
    'BAR': 'red',
    'CASA DE SHOW': 'purple',
    'DEPOSITO DE BEBIDAS': 'gray',
    'RESTAURANTE': 'blue',
    'CAFE': 'green',
    'CASA DE JOGOS': 'yellow',
    'OUTROS': 'black'
}

# Centralização do mapa
centro_lat = df['LATITUDE'].dropna().mean()
centro_lon = df['LONGITUDE'].dropna().mean()
centro = [centro_lat, centro_lon]

mapa = folium.Map(location=centro, zoom_start=13) # Aumentei o zoom para 13 para Belém abrir mais detalhada

# Loop pelos registros com coordenadas válidas
for _, row in df.dropna(subset=['LATITUDE', 'LONGITUDE']).iterrows():
    
    # 2. Descobre a categoria atual e busca a cor correspondente (padrão 'gray' se falhar)
    tipo = row['TIPO_ESTABELECIMENTO']
    cor_ponto = cores_categorias.get(tipo, 'gray')
    
    # 3. Cria um popup mais completo e elegante com HTML básico
    texto_popup = f"""
    <b>{row['DSC_ESTABELECIMENTO']}</b><br>
    <b>Tipo:</b> {tipo}<br>
    <b>Bairro:</b> {row['BAIRRO']}
    """
    
    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=5,                 # Tamanho 5 fica excelente para visualização urbana
        color=cor_ponto,          # Cor da borda
        fill=True,
        fill_color=cor_ponto,     # Cor do preenchimento
        fill_opacity=0.6,         # Transparência para dar destaque a pontos sobrepostos
        popup=folium.Popup(texto_popup, max_width=300) 
    ).add_to(mapa)

# Mostrar mapa
mapa